## Define libraries

In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [4]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# from helper import RAGHelper
from dotenv import load_dotenv
load_dotenv()
import os
# import pandas as pd
# import re
# import json

# import numpy as np
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import f1_score

In [5]:
# ingestion
# embedding_model - 2
# chunking_size - 256, 512, 1024
# chunking_overlap - 200, 100, 50
# search_type - 2
# search_k - 3, 5, 7
# vector_database - 3
# retrieval techniques - 2
# reranking
# gen_model - 2



## Define Variables

In [ ]:
# Constants

DATASET_SOURCE = 'rungalileo/ragbench'
DATASET_NAME = {"delucionqa":"Jeep manual", "emanual": "TV manual", "techqa":"Technotes"}
DATA_SPLIT = 'test'
VECTOR_DATABASE = 'chroma'
CHUNKING_SIZES = [256, 512, 1024]
CHUNKING_OVERLAPS = [50, 100, 200]
SEPARATORS = ["\n\n", "\n", " ", ".", ","]
DOMAIN = 'customer support'







In [7]:
### Parameters


# embedding
chromadb_folder = "../database"
# gen_model = "llama-3.1-8b-instant"
embedding_model = "BAAI/LLM-Embedder"

# db_name = f"finance_{chunking_size}_{chunking_overlap}"
# persist_directory = f"{chromadb_folder}/{db_name}"
collection_name = embedding_model.replace("/", "_") + "_cs"
embedding_function = HuggingFaceEmbeddings(model=embedding_model)

# search_type = "similarity"
# search_kwargs = {"k":3}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

f:\Github\Capstone\nextgen-rag-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sunil\.cache\huggingface\hub\models--BAAI--LLM-Embedder. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
def deduplicate_data(data, doc_type):
    data_dict = {}
    for d in data:
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            data_dict[document] = {"docid":[d["id"]]}
    for k in data_dict:
        data_dict[k]["document_type"] = doc_type
        
    return data_dict

## Data Fetching

In [ ]:
for k, v in DATASET_NAME.items():
    print(f"Creating embeddings for {k} using {embedding_model}")
    MAX_CHUNKS = 5000

    dataset = load_dataset(DATASET_SOURCE, k, split=DATA_SPLIT)
    dedup = deduplicate_data(dataset, v)
    docs = [
        Document(
            metadata=v, 
            page_content=k
        )
        for k, v in dedup.items()
    ]
    
    for cs, co in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):
        db_name = f"{DOMAIN}_{cs}_{co}"
        persist_directory = f"{chromadb_folder}/{db_name}"
            
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=co, separators=SEPARATORS)
        docs_chunks = text_splitter.split_documents(docs)
        print(len(docs_chunks))
        
        if len(docs_chunks) > MAX_CHUNKS:
            for i in range(0, len(docs_chunks), MAX_CHUNKS):
                doc_chunk = docs_chunks[i:i+MAX_CHUNKS]
        
                if os.path.exists(persist_directory) and os.listdir(persist_directory):
                    print(f"Loading existing vector database...{db_name}")
                    vector_db = Chroma(
                        collection_name=collection_name,
                        persist_directory=persist_directory, 
                        embedding_function=embedding_function
                    )
                    vector_db.add_documents(doc_chunk)
                else:
                    print(f"Creating new vector database...{db_name}")
                    vector_db = Chroma.from_documents(
                        collection_name=collection_name,
                        documents=doc_chunk, 
                        embedding=embedding_function, 
                        persist_directory=persist_directory
                    )
        else:
            # FIXED: This 'if' statement is now perfectly aligned with the 'else' below it
            if os.path.exists(persist_directory) and os.listdir(persist_directory):
                print(f"Loading existing vector database...{db_name}")
                vector_db = Chroma(
                    collection_name=collection_name,
                    persist_directory=persist_directory, 
                    embedding_function=embedding_function
                )
                vector_db.add_documents(docs_chunks)
            else:
                print(f"Creating new vector database...{db_name}")
                vector_db = Chroma.from_documents(
                    collection_name=collection_name,
                    documents=docs_chunks, 
                    embedding=embedding_function, 
                    persist_directory=persist_directory
                )

print(f"Data load into vector database completed for {embedding_model}")

Creating embeddings for delucionqa using BAAI/LLM-Embedder


Generating train split:   0%|          | 0/1460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/182 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/184 [00:00<?, ? examples/s]

2024
Creating new vector database...customer support_256_50
1026
Creating new vector database...customer support_512_100
528
Creating new vector database...customer support_1024_200
Creating embeddings for emanual using BAAI/LLM-Embedder


...

Generating train split:   0%|          | 0/1192 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/304 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/314 [00:00<?, ? examples/s]

16981
Loading existing vector database...customer support_256_50
Loading existing vector database...customer support_256_50
Loading existing vector database...customer support_256_50
Loading existing vector database...customer support_256_50
8436
Loading existing vector database...customer support_512_100
Loading existing vector database...customer support_512_100
3988
Loading existing vector database...customer support_1024_200
Data load into vector database completed for BAAI/LLM-Embedder


: 